# VRoid Companion Studio — Colab Demo Runner

Runs the app in Colab with a public Cloudflare tunnel URL. Session dies when Colab times out (~12h) — for production use HuggingFace Space or Docker.


In [ ]:
# 1) Clone your fork
!git clone https://github.com/YOUR_USER/vroid-companion-studio.git app && cd app && ls


In [ ]:
# 2) Install system deps + mongo
!apt-get update -qq && apt-get install -y -qq mongodb curl && mkdir -p /data/db
import subprocess; subprocess.Popen(['mongod', '--dbpath', '/data/db', '--fork', '--logpath', '/tmp/mongo.log'])


In [ ]:
# 3) Backend deps + secrets
import os
os.environ['MONGO_URL'] = 'mongodb://localhost:27017'
os.environ['DB_NAME'] = 'vroid_companion'
os.environ['EMERGENT_LLM_KEY'] = input('EMERGENT_LLM_KEY: ')
!pip install -q -r app/backend/requirements.txt httpx


In [ ]:
# 4) Build the frontend
!cd app/frontend && npm install -g yarn --silent && yarn install --silent && REACT_APP_BACKEND_URL='' yarn build


In [ ]:
# 5) Start backend + tunnel
import subprocess, time, re
subprocess.Popen(['python', 'app/entrypoint.py'])
time.sleep(6)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
import subprocess
p = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:7860'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end='')
    if 'trycloudflare.com' in line:
        break
